In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\tmp\\course-restructure-20261118\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0

# Разбор A. Прибор и специфичность

Даже идеальная электронная модель не устраняет шум, фон и перекрытие сигналов. Может ли UV отличить кислоту B от лактона? Без соответствующих спектров ответ неизвестен. На A проверим закон прохождения света на условных концентрациях, а ИК обсудим как дополнительный независимый канал идентификации. Гипотеза: рост stray light искажает линейность при большом поглощении.

Смоделируйте ε=10000 L mol^-1 cm^-1, l=1 cm, c=0…4e-4 mol/L; сравните s=0/.001. Покажите линейную и насыщенную кривые. На контрольных T=.1 и .01 рассчитайте A. NIST-ИК кофеина доступен отдельной загрузкой с проверкой hash; сырой файл с ограниченными правами не включается в публикацию. Параметрический тренажёр повторяет эту модель.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M08-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
import numpy as np
c=np.linspace(0,4e-4,101);ideal=10000*c
stray=.001;observed=-np.log10((10**(-ideal)+stray)/(1+stray))
assert np.allclose(-np.log10(10**(-ideal)),ideal)
print({'A_T_0.1':-math.log10(.1),'A_T_0.01':-math.log10(.01)})


{'A_T_0.1': 1.0, 'A_T_0.01': 2.0}


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
fig,ax=plt.subplots();ax.plot(c,ideal,label='ideal');ax.plot(c,observed,label='stray=0.001')
ax.set(xlabel='Simulated concentration / mol L^-1',ylabel='Absorbance');ax.legend();publish_plot(fig,'instrument.png')


<Figure size 640x480 with 1 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'kind':'simulation','epsilon_L_mol_cm':10000,'path_cm':1,'stray_fraction':stray,
        'max_deviation_A':float(max(ideal-observed)),'chemical_specificity':'NOT_RUN'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M08-L02','A',result,ROOT,inputs=[])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261118-full-course-windows\workspace\my_research\evidence\M08-L02\20261118T141612325902Z-A-a63e3d6b3ee2490fa08d6aba2ebfa5cf.json


Калибровка не устанавливает селективность в смеси. Для B нужны подходящие стандарты форм и ортогональная HPLC/LC-MS проверка. Этот вопрос непосредственно ведёт к анализу реакций и превращений формы во времени.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?